In [184]:
import pandas as pd
from validation_structure import normaliser, PAYS_COUVERTS
import re

**/!\ : `, dtype=str, keep_default_na=False` pour contraindre à considérer les données en *string* plutot qu'en *object***

In [185]:
df_tva = pd.read_csv("../data/numeros_tva.csv", dtype=str, keep_default_na=False)
# /!\ :, dtype=str, keep_default_na=False pour contraindre à considérer les données en <string> plutot qu'en <object>

df_tva

,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie
0,1,Nord Technologies GmbH,DK,DK73224645,2025-07-27,portail_client
1,2,Solaris Technologies SARL,PT,PT787327239,2024-03-29,reprise_erp
2,3,Verso Consulting NV,LU,LU71043234,2024-02-16,crm
3,4,Lumen Distribution Sp. z o.o.,SE,SE.751298146001,2024-02-27,import_fournisseur
4,5,Textil Services Oy,NL,NL505862176B88,2025-04-14,crm
...,...,...,...,...,...,...
9995,9996,Cobalt Trading A/S,PT,PT714332224,2025-01-19,import_fournisseur
9996,9997,Delta Trading Oy,FR,FR60801776699,2025-04-18,crm
9997,9998,Corvus Industries SARL,SE,SE96633532830,2024-09-12,reprise_erp
9998,9999,Prima Distribution AB,LU,LU.98154452,2024-02-15,reprise_erp


**1. Combien de pays distincts ? Combien de formats différents pour un même pays ?**

In [186]:
pays = {}

for i in df_tva["pays_declare"]:
    pays[i] = pays.get(i, 0) + 1

print(f"Nombre de pays : {len(pays)}")
pays

Nombre de pays : 15


{'DK': 961,
 'PT': 947,
 'LU': 956,
 'SE': 949,
 'NL': 947,
 'BE': 959,
 'FI': 902,
 'GB': 104,
 'FR': 974,
 'ZZ': 115,
 'PL': 939,
 'IT': 947,
 'UK': 104,
 'XX': 87,
 'QQ': 109}

In [187]:
df_tva["pays_declare"].value_counts()

pays_declare
FR    974
DK    961
BE    959
LU    956
SE    949
PT    947
NL    947
IT    947
PL    939
FI    902
ZZ    115
QQ    109
GB    104
UK    104
XX     87
Name: count, dtype: int64

In [188]:
df_tva["numero_tva_normaliser"] = df_tva["numero_tva"].map(normaliser)

In [189]:
def signature(s: str) -> str:
    return re.sub(r"[A-Za-z]", "A", re.sub(r"\d", "9", s))

In [190]:
df_tva["signature"] = df_tva["numero_tva"].map(signature)

In [191]:
df_tva

,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,numero_tva_normaliser,signature
0,1,Nord Technologies GmbH,DK,DK73224645,2025-07-27,portail_client,DK73224645,AA99999999
1,2,Solaris Technologies SARL,PT,PT787327239,2024-03-29,reprise_erp,PT787327239,AA999999999
2,3,Verso Consulting NV,LU,LU71043234,2024-02-16,crm,LU71043234,AA99999999
3,4,Lumen Distribution Sp. z o.o.,SE,SE.751298146001,2024-02-27,import_fournisseur,SE751298146001,AA.999999999999
4,5,Textil Services Oy,NL,NL505862176B88,2025-04-14,crm,NL505862176B88,AA999999999A99
...,...,...,...,...,...,...,...,...
9995,9996,Cobalt Trading A/S,PT,PT714332224,2025-01-19,import_fournisseur,PT714332224,AA999999999
9996,9997,Delta Trading Oy,FR,FR60801776699,2025-04-18,crm,FR60801776699,AA99999999999
9997,9998,Corvus Industries SARL,SE,SE96633532830,2024-09-12,reprise_erp,SE96633532830,AA99999999999
9998,9999,Prima Distribution AB,LU,LU.98154452,2024-02-15,reprise_erp,LU98154452,AA.99999999


In [192]:
df_tva[df_tva["pays_declare"].isin(PAYS_COUVERTS)].groupby("pays_declare")["signature"].nunique().sort_values(ascending=False)

pays_declare
NL    29
SE    29
IT    28
BE    28
PL    27
FI    26
FR    26
LU    25
DK    24
PT    24
Name: signature, dtype: int64

**2. Quelle proportion de lignes porte des caractères qui n'ont rien à faire dans un numéro de TVA ?**

In [193]:
nb_ln_raf_space = df_tva["numero_tva"].str.contains(" ", regex=False).sum()
nb_ln_raf_tiret = df_tva["numero_tva"].str.contains("-", regex=False).sum()
nb_ln_raf_uscore = df_tva["numero_tva"].str.contains("_", regex=False).sum()
nb_ln_raf_point = df_tva["numero_tva"].str.contains(".", regex=False).sum()
#print(f"Pct de lignes contenant des caractères n'ayant rien à faire dans un n° de tva : {round(nb_ln_raf / len(df_tva["numero_tva"]) * 100, 2)} %")
print(nb_ln_raf_space, nb_ln_raf_tiret, nb_ln_raf_uscore, nb_ln_raf_point)

938 471 0 404


In [194]:
df_tva["vide"] = df_tva["numero_tva_normaliser"].isin(["","NA","NULL"])
df_tva["parasite"] = (df_tva["numero_tva"] != df_tva["numero_tva_normaliser"]) & (df_tva["vide"] == False)

df_tva["parasite"].sum()

np.int64(2140)

**3. Combien de valeurs vides, et sous combien de formes différentes le vide se présente-t-il ?**

In [195]:
df_tva[df_tva["vide"]]["numero_tva"].value_counts()

numero_tva
         60
         59
-        55
N/A      48
null     38
NU.LL     1
Name: count, dtype: int64

In [196]:
df_tva[df_tva["vide"]]["numero_tva"].apply(repr).value_counts()

numero_tva
''         60
' '        59
'-'        55
'N/A'      48
'null'     38
'NU.LL'     1
Name: count, dtype: int64

**4. Le pays déclaré dans le CRM correspond-il toujours au préfixe du numéro ?**

In [197]:
#df_tva[df_tva["pays_declare"].isin(PAYS_COUVERTS)].groupby("pays_declare")["signature"].nunique().sort_values(ascending=False)
#df_tva[df_tva["source_saisie"] == "crm"]["numero_tva"].str.startswith(str(df_tva["pays_declare"]))
#df_tva[df_tva["source_saisie"] == "crm"]["numero_tva"].str[:2].equals(df_tva["pays_declare"])
#df_tva[df_tva["source_saisie"] == "crm"]["numero_tva"].str[:2] == df_tva[df_tva["source_saisie"] == "crm"]["pays_declare"]

df_tva["tout_bon"] = df_tva["numero_tva"].str[:2] == df_tva["pays_declare"]

display(df_tva[(df_tva["source_saisie"] == "crm") & (df_tva["tout_bon"] == False)])
display(df_tva[(df_tva["source_saisie"] == "crm") & (df_tva["tout_bon"] == False)]["id"].count())
display(df_tva[(df_tva["tout_bon"] == False)]["id"].count())


,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,numero_tva_normaliser,signature,vide,parasite,tout_bon
37,38,Granit Trading AB,FI,FI26606369,2024-11-30,crm,FI26606369,AA99999999,False,True,False
47,48,Granit Materials BV,NL,NL413400931B49,2025-02-26,crm,NL413400931B49,AA999999999A99,False,True,False
63,64,Ardent Foods Oy,DK,DK91970813,2025-03-29,crm,DK91970813,AA99999999,False,True,False
91,92,Ardent Industries Lda,FR,,2025-04-20,crm,,,True,False,False
107,108,Vela Materials SpA,NL,nl785427788b39,2024-04-18,crm,NL785427788B39,AA999999999A99,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...
9859,9860,Vertex Logistics AB,BE,BE1232211289,2024-05-24,crm,BE1232211289,AA9999999999,False,True,False
9939,9940,Vertex Trading Sp. z o.o.,DK,DK42075442,2024-08-29,crm,DK42075442,AA99999999,False,True,False
9954,9955,Prima Industries Lda,FR,72551923618,2025-02-03,crm,72551923618,99999999999,False,False,False
9955,9956,Lusitania Materials A/S,DK,63761613,2024-03-06,crm,63761613,99999999,False,False,False


np.int64(349)

np.int64(1645)

In [198]:
df_tva["tout_bon_ap_norm"] = df_tva["numero_tva_normaliser"].str[:2] == df_tva["pays_declare"]

display(df_tva[(df_tva["source_saisie"] == "crm") & (df_tva["tout_bon_ap_norm"] == False)])
display(df_tva[(df_tva["source_saisie"] == "crm") & (df_tva["tout_bon_ap_norm"] == False)]["id"].count())
display(df_tva[(df_tva["tout_bon_ap_norm"] == False)]["id"].count())

,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,numero_tva_normaliser,signature,vide,parasite,tout_bon,tout_bon_ap_norm
91,92,Ardent Industries Lda,FR,,2025-04-20,crm,,,True,False,False,False
134,135,Kappa Logistics Lda,NL,396904956B68,2025-04-01,crm,396904956B68,999999999A99,False,False,False,False
194,195,Prima Consulting Sp. z o.o.,SE,107327005801,2025-01-12,crm,107327005801,999999999999,False,False,False,False
259,260,Textil Trading Sp. z o.o.,LU,75041556,2025-03-06,crm,75041556,99999999,False,False,False,False
261,262,Corvus Distribution SpA,BE,0572894866,2024-08-10,crm,0572894866,9999999999,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
9712,9713,Kappa Materials AB,DK,82392963,2024-09-02,crm,82392963,99999999,False,False,False,False
9838,9839,Ardent Industries AB,PT,472620541,2024-05-13,crm,472620541,999999999,False,False,False,False
9850,9851,Vertex Industries A/S,NL,879062472B21,2025-03-07,crm,879062472B21,999999999A99,False,False,False,False
9954,9955,Prima Industries Lda,FR,72551923618,2025-02-03,crm,72551923618,99999999999,False,False,False,False


np.int64(180)

np.int64(793)

**5. Le CSV et l'Excel contiennent-ils vraiment la même chose une fois chargés ? Lequel prenez-vous comme source de vérité, et pourquoi ?**

In [199]:
df_tva_csv = pd.read_csv("../data/numeros_tva.csv", dtype=str, keep_default_na=False)

In [200]:
df_tva_xlsx = pd.read_excel("../data/numeros_tva.xlsx", dtype=str, keep_default_na=False)

In [201]:
df_diff = df_tva_csv.compare(df_tva_xlsx)

df_diff.notna().sum().sum()

np.float64(0.0)

- Pas de différences entre les 2 tableaux en terme de cellules.
- Grande différence cependant en terme de temps de chargement entre le csv et le xlsx : csv ~0.0s / xlsx ~3.2s

**6. Les canaux de saisie et les périodes de saisie se ressemblent-ils, ou certains sortent-ils du lot ?**

In [202]:
print(f"Première date saisie : {df_tva["date_saisie"].min()}")
print(f"Dernière date saisie : {df_tva["date_saisie"].max()}")

Première date saisie : 2024-01-01
Dernière date saisie : 2025-10-01


In [212]:
df_tva[df_tva["parasite"] | df_tva["vide"]][["id", "source_saisie"]].groupby("source_saisie").count()

,id
source_saisie,
crm,494
import_fournisseur,485
portail_client,454
reprise_erp,499
saisie_manuelle,469


- répartition des erreurs assez équilibrée entre les différentes sources de saisies